# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO

duplicates = df.duplicated().sum()
print(duplicates)

clean= df.drop_duplicates().copy()

log('duplicates', 'dropped exact duplicate rows', duplicates)

15
[duplicates] dropped exact duplicate rows (15 row(s))


I frist found the total number of duplcates and created a clean version of the df without the duplicates. Then I saved this function in the DECISIONS using log

### TODO 2 — clean `price` -> float

In [4]:
# TODO

clean['price'] = (
    clean['price']
    .astype(str)
    .str.replace('$', '', regex = False)
    .str.strip()
    .astype(float)
)

log('price', 'removed $ and made float', len(clean))

print(clean['price'].dtype)
print(clean['price'].head())

[price] removed $ and made float (300 row(s))
float64
0    12.0
1     7.5
2     7.5
3     7.5
4     7.5
Name: price, dtype: float64


I converted all of the strings to floats so that prices would be recongized as numerics and not text

### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO

clean ['qty'] = pd.to_numeric(clean['qty'], errors = 'coerce')

dirty_qty = clean['qty'].isna() | (clean['qty'] < 0)
removed_qty = dirty_qty.sum()

clean = clean.loc[~dirty_qty].copy()

log('quantity', 'dropped rows with missing/negative qty', removed_qty)




[quantity] dropped rows with missing/negative qty (25 row(s))


I first made all of the qty numeric to then find what the 'dirty_qty' was, the ones that had missing or negative numbers. I editted the clean df to have all the qty except the 'dirty_qty'.

### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse

print('Before:')
print(clean['item'].value_counts())

before_items = clean['item'].nunique()


ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho',
}

rows_changed_items = clean['item'].isin(['cheese burger', 'foam finger', 'rain poncho']
                                         ).sum()

clean['item'] = clean['item'].map(ITEM_MAP)

after_items = clean['item'].nunique()

print('\nAfter:')
print(clean['item'].value_counts())

log('items', f'collasped {before_items} item variants to {after_items} products',
    rows_changed_items
    )


Before:
item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64

After:
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64
[items] collasped 6 item variants to 3 products (126 row(s))


I first looked over the original items to find the spelling and then used ITEM MAP to collaspe the items. I then made 'items' in the clean df the collasped version.

### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
# TODO

print('Before:')
print(clean['category'].value_counts())

before_categories = clean['category'].nunique()

CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Merch',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear',
}

rows_changed_categories = clean['category'].isin(['food', 'Apparel', 'rain-gear']).sum()


clean['category'] = clean['category'].map(CATEGORY_MAP)

after_categories = clean['category'].nunique()

print('\nAfter:')
print(clean['category'].value_counts())

log('categories',
    f'normalized {before_categories} category variants to {after_categories} categories; combined Apperal with Merch', rows_changed_categories)

Before:
category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64

After:
category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64
[categories] normalized 6 category variants to 3 categories; combined Apperal with Merch (132 row(s))


I combined all of categories intended to display the same info that had different spelling (ex. rain-gear and Raingear) by using the category map function, and I made merch and apparel the same category under "Merch" to simplfy the listings. the 'clean' df was then updated with the new cateogries.

### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert clean.duplicated().sum() == 0
assert clean['qty'].min() >= 1
assert clean['price'].dtype == float
assert set(clean['item'].unique()) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
assert set(clean['category'].unique()) == {'Food', 'Merch', 'RainGear'}
print('clean:', clean.shape)

clean: (275, 5)


I added two assertion to make sure that I properly cleaned the item names in TODO 4 and the category names in TODO 6. All of the other checks passed, signifying that my TODOs have ran properly.

### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
# TODO

clean['revenue'] = clean['qty'] * clean['price']

revenue_by_category = clean.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)

overall_revenue = round(clean['revenue'].sum(), 2)

print('Revenue by category')
print(revenue_by_category)
print('\nOverall revenue:', overall_revenue)

Revenue by category
category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64

Overall revenue: 4740.0


I created a new column that displays revenue (qty * price), using groupby to group by category. I used ascending = False to have the highest revenue first. I used the .sum functioin to add together the revenue from each category to display the overall revenue.

**What I would tell the vendor:** _..._

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped exact duplicate rows,15
1,price,removed $ and made float,300
2,quantity,dropped rows with missing/negative qty,25
3,items,collasped 6 item variants to 3 products,126
4,categories,normalized 6 category variants to 3 categories...,132


The number of rows affected by each decision in the log matches the changes I observed from each step of the assignment.

## Revenue Analysis

In [11]:
rev_test = df.copy()
rev_test['price_num'] = pd.to_numeric(
    rev_test['price'].str.replace('$', '', regex=False),
    errors='coerce'
)

rev_test['qty_num'] = pd.to_numeric(rev_test['qty'], errors='coerce')

rev_test['revenue'] = rev_test['qty_num'] * rev_test['price_num']

rev_before_dupes = rev_test['revenue'].sum()
test_no_dupes = rev_test.drop_duplicates()
rev_after_dupes = test_no_dupes['revenue'].sum()
dupes_change = rev_before_dupes - rev_after_dupes

rev_before_qty = test_no_dupes['revenue'].sum()
dirty_qty_test = (
    test_no_dupes['qty_num'].isna() | (test_no_dupes['qty_num'] < 1)
)

rev_after_qty = test_no_dupes.loc[~dirty_qty_test]['revenue'].sum()
qty_change = rev_before_qty - rev_after_qty

print('DUPLICATE STEP')
print(f'Revenue before dropping duplicates: ${rev_before_dupes:.2f}')
print(f'Revenue after dropping duplicates: ${rev_after_dupes:.2f}')
print(f'Revenue change: ${dupes_change:.2f}')

print('QUANTITY STEP')
print(f'Revenue before dropping rows with missing/negative qty: ${rev_before_qty:.2f}')
print(f'Revenue after dropping rows with missing/negative qty: ${rev_after_qty:.2f}')
print(f'Revenue change: ${qty_change:.2f}')




DUPLICATE STEP
Revenue before dropping duplicates: $4852.50
Revenue after dropping duplicates: $4594.50
Revenue change: $258.00
QUANTITY STEP
Revenue before dropping rows with missing/negative qty: $4594.50
Revenue after dropping rows with missing/negative qty: $4740.00
Revenue change: $-145.50


I needed to calculate the revenue change of dropping duplicates and of dropping rows with negative/missing data so that I could answer part (A) of the write up. In this code, I made a copy of the df and then calculated the revenue before and after the duplicate drops and missing/negative drops to figure out which step of cleaning changed the revenue total the most.

### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) Dropping duplicates changed my total revenue the most.
- Revenue before the drop was $4852.50

- Revenue after was $4594.50

- Revenue, therefore, decreaseed by $258.00.

b) The decision I made to combine the categories of 'Apperal;' and 'Merch' into a single 'Merch' category. Someone else could have reasonably kepy 'Apperal' as its own category because combining the two does not change the total reveune. I wanted to combine the two because I thought it would look cleaner in the data frame to have a minimal number fo categories. But, this decision is not necessary, and was a choice of personal perference.